# Lab: Naive Bayes

## 1. Vì sao là "Naive" Bayes?

Naive Bayes là một họ thuật toán phân loại dựa trên định lý Bayes. Tên gọi *naive* (ngây thơ) đến từ một giả định lớn: ta giả sử các đặc trưng (feature) **độc lập có điều kiện** với nhau khi đã biết nhãn. Trong thực tế giả định này gần như luôn sai — nhưng kỳ lạ là Naive Bayes vẫn chạy rất tốt trên nhiều bài, đặc biệt là phân loại văn bản (spam, sentiment).

## 2. Định lý Bayes

$$
P(y \mid x) = \frac{P(x \mid y)\, P(y)}{P(x)}
$$

- $P(y)$: **prior** — xác suất tiên nghiệm của lớp $y$.
- $P(x \mid y)$: **likelihood** — xác suất quan sát thấy $x$ nếu nhãn là $y$.
- $P(x)$: **evidence** — xác suất quan sát thấy $x$ (nói chung).
- $P(y \mid x)$: **posterior** — cái ta muốn biết: xác suất nhãn là $y$ khi đã quan sát $x$.

Khi phân loại, mẫu số $P(x)$ giống nhau cho mọi lớp nên có thể bỏ:
$$
\hat{y} = \arg\max_y P(y) \cdot P(x \mid y)
$$

Đây là **Maximum A Posteriori (MAP)**.

## 3. Giả định Naive

Với $x = (x_1, x_2, \dots, x_n)$, ta giả sử:
$$
P(x \mid y) = \prod_{i=1}^{n} P(x_i \mid y)
$$

Nhờ giả định này, thay vì phải ước lượng phân phối liên kết $n$ chiều (số tham số bùng nổ), ta chỉ cần ước lượng $n$ phân phối một chiều — đơn giản hơn rất nhiều.

## 4. Ba biến thể phổ biến

Khác nhau ở cách mô hình hoá $P(x_i \mid y)$:

### 4.1. Bernoulli NB
Mỗi feature $x_i$ là binary (0/1). Phù hợp khi đặc trưng = "có/không có từ này trong văn bản".
$$
P(x_i \mid y) = p_{i,y}^{x_i} (1 - p_{i,y})^{1 - x_i}
$$

### 4.2. Multinomial NB
Mỗi feature $x_i$ là số đếm (count). Phù hợp khi đặc trưng = "từ này xuất hiện bao nhiêu lần" (TF, TF-IDF).
$$
P(x \mid y) \propto \prod_{i=1}^{n} p_{i,y}^{x_i}
$$

### 4.3. Gaussian NB
Mỗi feature $x_i$ liên tục, giả sử Gaussian:
$$
P(x_i \mid y) = \frac{1}{\sqrt{2\pi\sigma_{i,y}^2}}\exp\!\left(-\frac{(x_i - \mu_{i,y})^2}{2\sigma_{i,y}^2}\right)
$$

## 5. Laplace Smoothing

Vấn đề: nếu trong tập train không có mẫu nào có $x_i = 1$ với lớp $y$, thì $P(x_i = 1 \mid y) = 0$, kéo cả tích về 0 → mô hình chết một góc.

Giải pháp: cộng thêm $\alpha$ (thường $\alpha = 1$) vào tử và mẫu khi ước lượng xác suất. Đây là Laplace smoothing (a.k.a. add-one smoothing).

## 6. Ưu nhược điểm

**Ưu:** train cực nhanh, ít tham số, hoạt động tốt với dữ liệu cao chiều và nhỏ. Là baseline mạnh cho phân loại văn bản.

**Nhược:** giả định độc lập sai — khi feature có tương quan mạnh, xác suất ước lượng có thể méo. Tuy nhiên, nếu chỉ cần `argmax` (chứ không cần xác suất chính xác), kết quả vẫn ổn.

# THỰC HÀNH 1: Phân loại văn bản với Bernoulli & Multinomial NB

Dữ liệu: `Education.csv` — các câu nhận xét về giáo dục, gán nhãn `positive`/`negative`. Đây là bài phân loại văn bản nhị phân điển hình.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import BernoulliNB, MultinomialNB, GaussianNB
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_curve, auc)

np.random.seed(42)

df = pd.read_csv('Data/Education.csv')
print(f'Shape: {df.shape}')
print(f'Phân bố nhãn:\n{df["Label"].value_counts()}')
df.head()

In [ ]:
X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    df['Text'], df['Label'], test_size=0.2, random_state=42, stratify=df['Label'])

print(f'Train: {len(X_train_txt)}  Test: {len(X_test_txt)}')

### Hai cách biểu diễn văn bản

- **Bernoulli (binary)**: mỗi từ → 0/1 (có hay không có trong văn bản).
- **Multinomial (count)**: mỗi từ → số lần xuất hiện.

**Quan trọng**: chỉ `fit_transform` trên train, sau đó `transform` lên test — tránh data leakage (vocab của test "rò" vào train).

In [ ]:
# Bernoulli: binary features
vec_bin = CountVectorizer(binary=True, stop_words='english')
X_train_bin = vec_bin.fit_transform(X_train_txt)
X_test_bin  = vec_bin.transform(X_test_txt)

# Multinomial: count features
vec_cnt = CountVectorizer(stop_words='english')
X_train_cnt = vec_cnt.fit_transform(X_train_txt)
X_test_cnt  = vec_cnt.transform(X_test_txt)

print(f'Số feature (Bernoulli): {X_train_bin.shape[1]}')
print(f'Số feature (Multinomial): {X_train_cnt.shape[1]}')
print('5 từ ngẫu nhiên trong vocab:', np.random.choice(vec_cnt.get_feature_names_out(), 5).tolist())

In [ ]:
# Train cả hai
bnb = BernoulliNB()
bnb.fit(X_train_bin, y_train)
y_pred_bnb = bnb.predict(X_test_bin)

mnb = MultinomialNB()
mnb.fit(X_train_cnt, y_train)
y_pred_mnb = mnb.predict(X_test_cnt)

print(f'Bernoulli NB    accuracy: {accuracy_score(y_test, y_pred_bnb)*100:.2f}%')
print(f'Multinomial NB  accuracy: {accuracy_score(y_test, y_pred_mnb)*100:.2f}%')
print()
print('Báo cáo chi tiết — Multinomial NB:')
print(classification_report(y_test, y_pred_mnb))

### ROC curve

ROC vẽ True Positive Rate (recall) theo False Positive Rate khi quét ngưỡng. AUC = diện tích dưới đường cong, càng gần 1 càng tốt; 0.5 = đoán mò.

Lưu ý: **AUC < 0.5 nghĩa là model tệ hơn cả đoán mò**, nhưng có thể "đảo ngược" predict (lật label) để được > 0.5 — chứ không phải không cứu được.

In [ ]:
# Lấy xác suất class "positive" để vẽ ROC
pos_idx = list(mnb.classes_).index('positive')
proba_mnb = mnb.predict_proba(X_test_cnt)[:, pos_idx]

fpr, tpr, _ = roc_curve(y_test, proba_mnb, pos_label='positive')
auc_score = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'Multinomial NB (AUC = {auc_score:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Đoán mò (AUC = 0.5)')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

### Từ nào quan trọng nhất với mỗi lớp?

Multinomial NB lưu `feature_log_prob_[class][token]` = log P(token | class). Tỷ số log giữa hai lớp cho biết từ nào "thiên vị" về phía nào.

In [ ]:
vocab = vec_cnt.get_feature_names_out()
log_p_pos = mnb.feature_log_prob_[pos_idx]
log_p_neg = mnb.feature_log_prob_[1 - pos_idx]
log_ratio = log_p_pos - log_p_neg     # >0 → ngả về positive, <0 → ngả về negative

top_pos = np.argsort(log_ratio)[-10:][::-1]
top_neg = np.argsort(log_ratio)[:10]

print('Top 10 từ thiên vị POSITIVE:')
for i in top_pos:
    print(f'  {vocab[i]:20s}  log-ratio = {log_ratio[i]:+.3f}')
print('\nTop 10 từ thiên vị NEGATIVE:')
for i in top_neg:
    print(f'  {vocab[i]:20s}  log-ratio = {log_ratio[i]:+.3f}')

# THỰC HÀNH 2: Gaussian NB trên dữ liệu thuốc (drug200)

Dữ liệu `drug200.csv` có cả feature liên tục (Age, Na_to_K) và rời rạc (Sex, BP, Cholesterol). Mục tiêu: dự đoán loại thuốc phù hợp.

**Lưu ý quan trọng**: Gaussian NB *giả định feature là liên tục, phân phối chuẩn*. Nếu ép feature one-hot (binary) vào Gaussian NB, ta đang vi phạm giả định — kết quả vẫn chạy nhưng không phải là cách dùng đúng. Trong bài này ta dùng Gaussian NB trên feature đã encode chỉ để minh hoạ, kèm thảo luận giới hạn.

In [ ]:
from sklearn.preprocessing import LabelEncoder

drug = pd.read_csv('Data/drug200.csv')
print(drug.head())
print(f'\nShape: {drug.shape}, classes: {drug["Drug"].unique()}')

In [ ]:
# Encode các cột rời rạc
df_enc = drug.copy()
for col in ['Sex', 'BP', 'Cholesterol']:
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col])

X = df_enc.drop('Drug', axis=1).values
y = LabelEncoder().fit_transform(df_enc['Drug'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

gnb = GaussianNB()
gnb.fit(X_train, y_train)
y_pred = gnb.predict(X_test)
print(f'Gaussian NB accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%')
print()
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
drug_names = LabelEncoder().fit(df_enc['Drug']).classes_
ax.set_xticks(range(5)); ax.set_yticks(range(5))
ax.set_xticklabels(drug_names); ax.set_yticklabels(drug_names)
ax.set_xlabel('Dự đoán'); ax.set_ylabel('Thật')
ax.set_title('Confusion matrix — Gaussian NB trên drug200')
for i in range(5):
    for j in range(5):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.colorbar(im); plt.tight_layout(); plt.show()

## Tổng kết

1. **Định lý Bayes** + giả định độc lập có điều kiện = Naive Bayes.
2. Ba biến thể chính: Bernoulli (binary), Multinomial (count), Gaussian (liên tục).
3. **Laplace smoothing** ($\alpha$) giúp tránh xác suất = 0.
4. Trong bài text, Multinomial NB thường tốt hơn Bernoulli NB nếu document đủ dài.
5. Naive Bayes là *baseline cực mạnh* — train rất nhanh, kết quả ổn, nên luôn thử trước khi dùng model phức tạp.

## Tránh các bẫy
- KHÔNG `fit_transform` trên test — chỉ `transform`.
- KHÔNG fit Gaussian NB trên feature one-hot mà không cảnh báo về giả định bị vi phạm.
- KHI dataset nhỏ, accuracy trên test rất nhiễu — nên dùng cross-validation.

# BÀI TẬP VỀ NHÀ

## Bài 1: Cross-validation cho text classifier
Dataset Education chỉ có 52 dòng — accuracy trên 1 lần chia rất nhiễu. Hãy:
1. Dùng `StratifiedKFold(n_splits=5)` để cross-validate cả Bernoulli NB và Multinomial NB.
2. Báo cáo accuracy trung bình ± std cho mỗi model.
3. Model nào ổn định hơn?

*Gợi ý:* `from sklearn.model_selection import cross_val_score; cross_val_score(model, X, y, cv=5)`.

## Bài 2: Ảnh hưởng của Laplace smoothing
Train Multinomial NB trên Education với `alpha ∈ {0.01, 0.1, 1, 10, 100}`. Vẽ accuracy theo alpha. Quan sát: alpha quá nhỏ và quá lớn đều tệ — vì sao?

*Gợi ý:* `MultinomialNB(alpha=...)`. Alpha lớn = smoothing mạnh = mô hình "quên" data.

## Bài 3: TF-IDF thay vì raw count
Thay `CountVectorizer` bằng `TfidfVectorizer`. Train Multinomial NB. So sánh accuracy với raw count. TF-IDF có giúp không?

## Bài 4: CategoricalNB cho drug200
Sklearn có `CategoricalNB` — biến thể *đúng* cho feature rời rạc. Dùng nó trên drug200 (sau khi encode). So sánh với Gaussian NB. Cái nào tốt hơn? Vì sao?

*Gợi ý:* `from sklearn.naive_bayes import CategoricalNB`. Lưu ý: feature liên tục Age, Na_to_K cần discretize trước (`pd.cut` hoặc `KBinsDiscretizer`).

## Bài 5: Phân tích sai
Trên drug200, in ra 5 mẫu Gaussian NB đoán sai. Đặc trưng của chúng có gì đặc biệt? (Age cực, Na_to_K cao bất thường?)

*Gợi ý:* `mask = y_pred != y_test; X_test[mask][:5]`.